## LLM + Prompting

El objetivo de esta sección es utilizar un LLM para generar resúmenes automáticos a partir de los resultados obtenidos en el análisis exploratorio y modelado de churn y para esto se diseñaron dos prompts distintos con enfoques diferentes para posteriormente comparar los resultados y documentar estrategias para reducir riesgos de alucinación.

#### Prompt 1 — Técnico y analítico

Este prompt busca generar un resumen técnico del proyecto utilizando métricas reales del modelo, hallazgos del EDA y limitaciones detectadas durante el análisis.

In [5]:
prompt_1 = """
Eres un analista de datos senior.

El siguiente es un resumen de resultados de un análisis de churn sobre un dataset de 2,020
usuarios con variables de comportamiento en ventana rolling de 30 días.

Variables:
- age
- country
- events_last_30d
- purchases_last_30d
- total_spend_last_30d

Target:
- churned (9.5% positivo)

Resultados del modelo:
- Logistic Regression (AUC 0.509, F1 macro 0.42)
- Random Forest (AUC 0.491, F1 macro 0.51)

Importancia de variables en Random Forest:
- spend (29%)
- age (27%)
- events (21%)
- purchases (15%)
- country (7%)

Hallazgos:
- Outliers de dominio (edades negativas/200)
- Outliers extremos en eventos (hasta 900,000)
- Gastos negativos (-300)

Escribe un resumen ejecutivo de 5 párrafos con:
- hallazgos clave,
- limitaciones,
- y próximos pasos.
"""

print(prompt_1)


Eres un analista de datos senior.

El siguiente es un resumen de resultados de un análisis de churn sobre un dataset de 2,020
usuarios con variables de comportamiento en ventana rolling de 30 días.

Variables:
- age
- country
- events_last_30d
- purchases_last_30d
- total_spend_last_30d

Target:
- churned (9.5% positivo)

Resultados del modelo:
- Logistic Regression (AUC 0.509, F1 macro 0.42)
- Random Forest (AUC 0.491, F1 macro 0.51)

Importancia de variables en Random Forest:
- spend (29%)
- age (27%)
- events (21%)
- purchases (15%)
- country (7%)

Hallazgos:
- Outliers de dominio (edades negativas/200)
- Outliers extremos en eventos (hasta 900,000)
- Gastos negativos (-300)

Escribe un resumen ejecutivo de 5 párrafos con:
- hallazgos clave,
- limitaciones,
- y próximos pasos.



#### Output generado

In [7]:
output_1 = """
El análisis exploró un dataset de comportamiento de usuarios con el objetivo de predecir churn. 
La tasa de churn del 9.5% refleja un desbalance típico que requiere estrategias específicas de modelado. 
Se identificaron tres categorías de outliers relevantes: errores de dominio en edad (valores negativos y mayores de 100),
anomalías extremas en eventos que sugieren actividad automatizada o de bots, y gastos negativos que podrían corresponder 
a devoluciones. Cada tipo fue tratado con una estrategia diferenciada según su naturaleza.

Los modelos entrenados (Regresión Logística y Random Forest) presentaron AUC cercano a 0.5, indicando que las variables 
disponibles tienen poder predictivo limitado en su forma actual. El gasto total y 
la edad fueron las variables de mayor importancia, seguidas por el nivel de actividad (eventos). 
La variable país aportó la menor señal, sugiriendo que el comportamiento de churn es relativamente homogéneo entre los mercados
analizados.

La principal limitación identificada es la ausencia de variables contextuales clave: historial de soporte, tipo de plan, 
canal de adquisición y tendencias temporales. Adicionalmente, la naturaleza rolling del cálculo de features introduce riesgos 
de data leakage que deben resolverse antes de desplegar el modelo. Se detectaron registros duplicados (mismo user_id con mismos 
valores) que confirman este riesgo.

Para mejorar la capacidad predictiva, se recomienda: enriquecer las features con variables de tendencia y recency, aplicar 
validación temporal en lugar de split aleatorio, y explorar modelos de gradiente boosting con calibración de probabilidades. 
La definición del umbral de clasificación debe alinearse con el costo de negocio de una intervención incorrecta (falso positivo) 
vs. perder un cliente (falso negativo).

Como siguiente paso inmediato se sugiere una revisión del proceso ETL para resolver duplicados y confirmar la definición
temporal del target, seguido de un ciclo de feature engineering de 5–7 días antes de re-entrenar.
"""

print(output_1)


El análisis exploró un dataset de comportamiento de usuarios con el objetivo de predecir churn. 
La tasa de churn del 9.5% refleja un desbalance típico que requiere estrategias específicas de modelado. 
Se identificaron tres categorías de outliers relevantes: errores de dominio en edad (valores negativos y mayores de 100),
anomalías extremas en eventos que sugieren actividad automatizada o de bots, y gastos negativos que podrían corresponder 
a devoluciones. Cada tipo fue tratado con una estrategia diferenciada según su naturaleza.

Los modelos entrenados (Regresión Logística y Random Forest) presentaron AUC cercano a 0.5, indicando que las variables 
disponibles tienen poder predictivo limitado en su forma actual. El gasto total y 
la edad fueron las variables de mayor importancia, seguidas por el nivel de actividad (eventos). 
La variable país aportó la menor señal, sugiriendo que el comportamiento de churn es relativamente homogéneo entre los mercados
analizados.

La principal limi

### Prompt 2 — Orientado a negocio

Este prompt busca traducir los resultados técnicos a un lenguaje ejecutivo y accionable para stakeholders no técnicos.

In [8]:
prompt_2 = """
Imagina que eres el jefe de retención de clientes de una empresa de tecnología en Latinoamérica.

Tu equipo de datos analizó 2,020 clientes y encontró que el 9.5% abandonó el servicio.

Los modelos de predicción aún no son confiables (precisión cercana al azar), pero se sabe que los clientes que gastan menos 
y tienen menor actividad en la plataforma tienen mayor riesgo.

También se encontraron datos sucios:
- clientes con edades imposibles,
- niveles de actividad sospechosamente altos (posiblemente bots).

Escribe un correo de 3 párrafos para tu VP:
- resumiendo el estado del análisis,
- y solicitando 7 días adicionales para mejorar el modelo,
- incluyendo justificación de negocio.
"""

print(prompt_2)


Imagina que eres el jefe de retención de clientes de una empresa de tecnología en Latinoamérica.

Tu equipo de datos analizó 2,020 clientes y encontró que el 9.5% abandonó el servicio.

Los modelos de predicción aún no son confiables (precisión cercana al azar), pero se sabe que los clientes que gastan menos 
y tienen menor actividad en la plataforma tienen mayor riesgo.

También se encontraron datos sucios:
- clientes con edades imposibles,
- niveles de actividad sospechosamente altos (posiblemente bots).

Escribe un correo de 3 párrafos para tu VP:
- resumiendo el estado del análisis,
- y solicitando 7 días adicionales para mejorar el modelo,
- incluyendo justificación de negocio.



#### Output generado

In [9]:
output_2 = """
Estimado/a VP,

Quiero compartirte el estado actual del proyecto de predicción de churn. Analizamos el comportamiento de 2,020 clientes 
en los últimos 30 días y encontramos que aproximadamente un 9.5% abandonó el servicio. Observamos que los usuarios con menor 
gasto y menor actividad presentan mayor riesgo de churn, lo que ya nos permite identificar patrones iniciales para estrategias 
de retención.

Sin embargo, el modelo predictivo aún no alcanza un nivel de confiabilidad suficiente para activaciones automáticas. 
Detectamos problemas de calidad de datos, incluyendo edades inválidas y actividad extremadamente alta que podría corresponder a 
bots o errores de tracking. Estos problemas impactan directamente la capacidad predictiva y aumentan el riesgo de tomar 
decisiones incorrectas sobre clientes reales.

Solicito 7 días adicionales para fortalecer el modelo mediante limpieza de datos, incorporación de variables históricas y 
validación temporal adecuada. Estas mejoras permitirán construir un sistema más robusto y útil para campañas de retención 
futuras, reduciendo potencialmente costos asociados a pérdida de clientes.
"""

print(output_2)


Estimado/a VP,

Quiero compartirte el estado actual del proyecto de predicción de churn. Analizamos el comportamiento de 2,020 clientes 
en los últimos 30 días y encontramos que aproximadamente un 9.5% abandonó el servicio. Observamos que los usuarios con menor 
gasto y menor actividad presentan mayor riesgo de churn, lo que ya nos permite identificar patrones iniciales para estrategias 
de retención.

Sin embargo, el modelo predictivo aún no alcanza un nivel de confiabilidad suficiente para activaciones automáticas. 
Detectamos problemas de calidad de datos, incluyendo edades inválidas y actividad extremadamente alta que podría corresponder a 
bots o errores de tracking. Estos problemas impactan directamente la capacidad predictiva y aumentan el riesgo de tomar 
decisiones incorrectas sobre clientes reales.

Solicito 7 días adicionales para fortalecer el modelo mediante limpieza de datos, incorporación de variables históricas y 
validación temporal adecuada. Estas mejoras permitirán 

### Comparación de resultados

El Prompt 1 generó un resumen técnico más detallado y orientado a analistas de datos. Incluyó métricas, limitaciones del modelo, riesgos de data leakage y recomendaciones de mejora técnica. Por otro lado, el Prompt 2 produjo un resultado más ejecutivo y accionable, traduciendo hallazgos técnicos a lenguaje de negocio y enfocándose en impacto operacional y toma de decisiones.

La elección del prompt óptimo depende de la audiencia:

- equipos técnicos seria el Prompt 1,
- liderazgo y stakeholders el Prompt 2.

En un entorno real, ambos enfoques serían complementarios dentro del mismo proyecto.

###  Estrategias para evitar alucinaciones

Para reducir el riesgo de respuestas incorrectas o inventadas por parte del LLM, se aplicaron las siguientes estrategias:

- incluir métricas y resultados reales dentro del prompt,
- restringir explícitamente el uso de información externa,
- proporcionar contexto estructurado,
- limitar inferencias no soportadas por los datos,
- y validar manualmente el output generado contra los resultados obtenidos en Python.

Adicionalmente, prompts más específicos y estructurados reducen significativamente el riesgo de alucinación frente a prompts ambiguos o demasiado abiertos.